# Notebook 08 — Per-Series Raw Data Visualization

This notebook produces individual time-series plots for every raw macro series
(multpl.com + FRED) loaded during ingestion (step 1). Use it for:

- Visual QC of scraped data before feature engineering
- Spot-checking gap patterns and coverage
- Overlaying market_code regimes on raw series

**Prerequisites:** Run the pipeline through at least step 1.
```
python run_pipeline.py --steps 1
```

In [1]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
sys.path.insert(0, str(repo_root / "src"))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from market_regime import DATA_DIR, OUTPUT_DIR
from market_regime.config import load
from market_regime import plotting

cfg = load()
raw = pd.read_parquet(DATA_DIR / "raw" / "macro_raw.parquet")
print(f"Raw data: {raw.shape}  —  {raw.index[0]} to {raw.index[-1]}")

Raw data: (305, 54)  —  1950-03-31 00:00:00 to 2026-03-31 00:00:00


In [2]:
# ── Coverage heatmap: at a glance, which series cover which eras ───────────
from market_regime.runtime import RunConfig

run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)
plotting.plot_raw_series_coverage(raw, run_cfg)

/Users/glestryc/personal/github_repos/claude-scratch-work/src/market_regime/plotting.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ── Helper: plot one series with optional market_code overlay ─────────────
has_market_code = "market_code" in raw.columns

def plot_series(col: str, title: str | None = None, ax=None) -> None:
    """Plot one raw series, optionally shaded by market_code."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(14, 3))

    series = raw[col].dropna()
    ax.plot(series.index, series.values, linewidth=1.4, color="#1f3d7a")

    if has_market_code:
        mc = raw["market_code"].dropna()
        unique_codes = sorted(mc.astype(int).unique())
        for code in unique_codes:
            mask = mc.astype(int) == code
            for idx in mc.index[mask]:
                ax.axvspan(
                    idx,
                    idx + pd.tseries.frequencies.to_offset("QE"),
                    alpha=0.15,
                    color=plotting.CUSTOM_COLORS[code % len(plotting.CUSTOM_COLORS)],
                    linewidth=0,
                )

    ax.set_title(title or col, fontsize=9)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

    if standalone:
        fig.tight_layout()
        plt.show()
        plt.close(fig)

In [4]:
# ── Key series grid (quick overview) ─────────────────────────────────────
KEY_SERIES = [
    ("sp500",       "S&P 500 Price Level"),
    ("fred_gdp",    "US GDP (FRED, nominal)"),
    ("us_infl",     "US Inflation Rate"),
    ("10yr_ustreas", "10-Year Treasury Rate"),
    ("div_yield",   "S&P 500 Dividend Yield"),
    ("fred_baa",    "BAA Corporate Bond Yield"),
    ("cape_shiller","Shiller CAPE Ratio"),
    ("credit_spread" if "credit_spread" in raw.columns else "fred_baa",
     "Credit Spread (BAA−AAA)"),
]
KEY_SERIES = [(c, t) for c, t in KEY_SERIES if c in raw.columns]

n = len(KEY_SERIES)
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
if n == 1:
    axes = [axes]
for ax, (col, title) in zip(axes, KEY_SERIES):
    plot_series(col, title, ax=ax)

axes[-1].set_xlabel("Quarter")
fig.suptitle("Key Macro Series — Raw Data", fontsize=13, y=1.01)
fig.tight_layout()

out = OUTPUT_DIR / "plots" / "08_key_series.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_key_series.png


/Users/glestryc/tmp/ipykernel_44586/2419472829.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── All FRED series in one grid ───────────────────────────────────────────
fred_cols = [c for c in raw.columns if c.startswith("fred_")]
print(f"FRED series: {fred_cols}")

n = len(fred_cols)
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
if n == 1:
    axes = [axes]
for ax, col in zip(axes, fred_cols):
    plot_series(col, ax=ax)
axes[-1].set_xlabel("Quarter")
fig.suptitle("FRED Series — Raw Data", fontsize=13, y=1.01)
fig.tight_layout()

out = OUTPUT_DIR / "plots" / "08_fred_series.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

FRED series: ['fred_gdp', 'fred_gnp', 'fred_baa', 'fred_aaa', 'fred_cpi', 'fred_gs10', 'fred_tb3ms']
Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_fred_series.png


/Users/glestryc/tmp/ipykernel_44586/2966280153.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── All multpl series — paginated into groups of 10 ──────────────────────
multpl_cols = [c for c in raw.columns if not c.startswith("fred_") and c != "market_code"]
PAGE_SIZE = 10

for page_idx, start in enumerate(range(0, len(multpl_cols), PAGE_SIZE)):
    page = multpl_cols[start : start + PAGE_SIZE]
    n = len(page)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, page):
        plot_series(col, ax=ax)
    axes[-1].set_xlabel("Quarter")
    fig.suptitle(f"multpl.com Series — Page {page_idx + 1}", fontsize=13, y=1.01)
    fig.tight_layout()
    out = OUTPUT_DIR / "plots" / f"08_multpl_p{page_idx + 1:02d}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved {out}")
    plt.show()
    plt.close(fig)

Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_multpl_p01.png


/Users/glestryc/tmp/ipykernel_44586/1424888347.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_multpl_p02.png


/Users/glestryc/tmp/ipykernel_44586/1424888347.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_multpl_p03.png


/Users/glestryc/tmp/ipykernel_44586/1424888347.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_multpl_p04.png


/Users/glestryc/tmp/ipykernel_44586/1424888347.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_multpl_p05.png


/Users/glestryc/tmp/ipykernel_44586/1424888347.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Gap summary: which series have the most missing quarters ──────────────
feature_cols = [c for c in raw.columns if c != "market_code"]
nan_pct = raw[feature_cols].isna().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(4, len(nan_pct) * 0.25)))
colors = ["#cc3300" if v > 0.5 else "#3366cc" for v in nan_pct.values]
ax.barh(range(len(nan_pct)), nan_pct.values * 100, color=colors, edgecolor="none")
ax.set_yticks(range(len(nan_pct)))
ax.set_yticklabels(nan_pct.index, fontsize=7)
ax.set_xlabel("% quarters missing")
ax.set_title("Raw Series — Missing Data Summary", fontsize=12)
ax.axvline(50, color="#cc3300", linewidth=1, linestyle="--", label="50% missing")
ax.legend(fontsize=8)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

out = OUTPUT_DIR / "plots" / "08_missing_data_summary.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved {out}")
plt.show()

Saved /Users/glestryc/personal/github_repos/claude-scratch-work/outputs/plots/08_missing_data_summary.png


/Users/glestryc/tmp/ipykernel_44586/4186842195.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
